# Licensing recommendation analysis

This notebook turns the profile and exploratory analysis into a repeatable recommendation for Kwame Boateng’s fictional $14M back-catalog licensing decision. See the [lab brief](../data/lab_brief.md) and [decision record](../docs/DECISIONS.md).

## Decision questions

1. Does the evidence favor franchise bundles or standalone films, and is that conclusion stable across the required time windows?
2. Which eligible franchises and standalone titles merit consideration in each window, and what supports their inclusion?
3. Do directors or leading-cast members show repeatable associations with stronger outcomes after relevant context is considered?
4. What relative-value case can be made for the strongest franchise, and which KPI most strongly supports it?

## Expected deliverables

- A franchise-versus-standalone comparison reported separately for all years and 2000 onward.
- Side-by-side shortlists targeting three franchises and five standalones per window, with reasons, evidence limits, and rank changes.
- A cautious director and leading-cast signal analysis with role definitions, title counts, and uncertainty.
- A one-paragraph deal note for the strongest franchise and a one-page client memo covering the recommendation, shortlists, both windows, and limitations.

## Interpretation guardrails

Historical box office measures theatrical performance. Popularity changes over time and overlaps with vote count. Genre, talent, and collection patterns are observational. Keep plausible extremes in the primary analysis, and highlight any change in the recommendation between windows.

## What has been done so far

### Data-quality profile — `02_profile_movie_data.ipynb`

- The normalized tables and relationship grains were profiled for the current 112-movie cache.
- No exact raw-object duplicates, exact duplicate movie rows, or title/release-year semantic duplicate candidates were found. Tested range and date checks passed, while the profile also records missing reference labels, constant fields, and narrow language coverage.
- A copy-only dtype experiment preserved the normalized movie frame. Its memory result applies only to this small snapshot; runtime was not benchmarked.

### Exploratory analysis — `03_eda_movie_data.ipynb`

- Numeric distributions, IQR-flagged observations, and plausible extreme values were investigated. Popularity remains snapshot-sensitive.
- Associations were observed between budget and revenue, popularity and vote count, and vote count and rating. These are not causal findings; popularity and vote count overlap.
- Genre, collection, director, and top-billed-cast relationships were explored. Common genre indicators did not improve held-out R-squared over the budget/year/collection controls in this sample. Collection and talent residuals remain descriptive and sample-sensitive.

### Candidate coverage so far

The 127 input rows reconcile to 112 distinct cached titles, 6 repeated rows, 5 TMDB 404s, 3 invalid IDs, and 1 blank. Of the 112 cached titles, 71 were released in 2000 or later. The full pool has 16 titles without a collection ID, and the newer window has 8, before scoring eligibility. See the [findings log](../docs/findings.md).

**Current status**

The profile and EDA are complete for this snapshot. Step 1 below rebuilds the current candidate pool and checks both release windows. Scoring, comparison, shortlists, and the memo are next in the plan.

## Implementation plan

### 1. Reconcile inputs and apply the two release windows (implemented below)

- Reconcile every row of the current `id_list.csv` to a cache or fetch outcome. Intersect cached titles with the valid unique IDs in the current list so a previous quarter's cache entries cannot enter the candidate pool. Keep one row per `movie_id`. Report source losses, titles outside the window, and KPI eligibility exclusions separately.
- Use dated, released titles in **A — all years** and only dated, released titles with `release_date >= 2000-01-01` in **B — 2000 onward**. Report missing dates and release statuses. Every summary and score in B uses only its in-window titles. Both windows use the same scoring constants, eligibility gates, and tie-break rules. The windows overlap, so agreement measures sensitivity to the cutoff.
- Report counts by year or decade, title type, and collection, then compare outcome coverage and missingness. Use release year or era for context in nominal financial comparisons. The cutoff addresses Yaw's concern and leaves the historical dollar values nominal.

### 2. Use a fixed audience-first scoring rule

Use vote-adjusted audience rating as the primary shortlist KPI. It measures how TMDB voters rated a title, with fewer-vote ratings pulled toward a common reference. Vote count measures the breadth of that evidence and also serves as the reach-first sensitivity. Retain popularity as context. An audience-first shortlist could favor well-liked films with limited reach, which the reach-first comparison should surface.

- For **title eligibility**, require a finite `vote_average` in [0, 10] and an integer `vote_count` of at least 1,000. Report missing and invalid vote metrics, and titles below the threshold, as separate reasons. Apply financial validity rules to the corresponding KPIs.
- Calculate the **title score** as `(vote_count * vote_average + 1000 * 7.0) / (vote_count + 1000)`. Use 7.0 as a conservative point on the rating scale for this pool, whose exploratory mean is about 7.59. Make observed votes and prior votes count equally at the 1,000-vote entry threshold. These fixed lab assumptions apply across both windows and reruns. Use the full-precision score to rank and round for display.
- Order **standalones** by adjusted rating descending, vote count descending, then numeric `movie_id` ascending. A missing collection ID defines a standalone candidate in the available metadata.
- For **franchise eligibility and ranking**, group by `collection_id` within the window. Require at least two observed dated, released installments and score every such installment to qualify. A known in-window installment with unusable rating evidence makes the proxy bundle ineligible. Rank by median adjusted rating descending, lowest installment adjusted rating descending, median vote count descending, then numeric `collection_id` ascending. Each observed installment gets equal weight. List the observed titles and flag unknown coverage beyond the candidate list. Use these observed titles as the hypothetical lab package.
- For the **shortlists**, take up to three eligible collections and five eligible standalones in order, and explain any shortfall. Calculate positive-budget, positive-gross financial proxies where defined: reported gross minus budget, and `(revenue - budget) / budget`. Log zeros separately from missing values. Show these as context for historical performance, with the valid-film count. TMDB provides no platform outcomes or acquisition costs from which to calculate licensing return.

### 3. Compare bundles and standalones at both levels

- Compare like decision units first. Each eligible proxy bundle contributes its median adjusted rating once, and each eligible standalone contributes its adjusted rating once. Report the gap in group medians in rating points, the spread, and the number of eligible units and films. Also show a per-film comparison of all score-eligible collection-linked titles and standalones, so the effect of bundle eligibility and large collections is visible.
- If each side has at least five eligible units, sort decision units by ID within each side. Use NumPy’s default random generator with seed 42 and draw 2,000 bootstrap resamples within each side. Take the 2.5th and 97.5th percentiles of the bundle-minus-standalone median gaps for a descriptive 95% interval. The primary comparison favors bundles if the interval lies entirely above +0.25 rating points, or standalones if it lies entirely below −0.25. Otherwise report insufficient evidence to favor either. The 0.25-point margin and minimum group size are lab judgments. The interval reflects variation within this candidate list.
- Retain plausible extreme values in the primary analysis. Show bundle size, lowest installment score, valid-financial-film counts, and observed gross totals alongside per-film medians. Report the impact of removing the highest-scoring installment from each bundle (lowest numeric movie ID in a score tie) in a sensitivity run, including any bundle that drops below two films.

### 4. Review genre and talent signals

- Count titles per genre and report genre overlaps. Use the EDA cutoffs in both windows. Require at least 10 titles for descriptive comparisons and 15 for genre indicators in an adjusted model. Examine relationships with audience response as well as exploratory log gross. Use budget, release year, genre, and collection status as context where the sample supports it.
- Identify directors by crew job `Director` and use cast order 0–2 as the leading-cast proxy. Deduplicate on `(movie_id, person_id, role)`, group people and collections by ID, and attach names for display. Report distinct titles, distinct collections, and distinct standalone titles. Count each collection as one group and each standalone as its own group. Require at least three distinct titles across at least two groups for a cross-title signal. Flag one-franchise concentrations even when the title threshold is met.
- Compare adjusted outcomes by person and check how the direction changes when the person's largest group by title count is omitted. Break size ties by group type (collection before standalone) and then numeric group ID. If fewer than two titles remain, or the direction flips or disappears, label the signal fragile and show why. Report changes in eligibility and signal between windows. Use release-time and collection-aware holdouts if making any predictive claim. With sparse data, keep the result descriptive and avoid causal language.

### 5. Stress-test and make the rankings reproducible

- Rerun both windows with a reach-first order. Lead with vote count for titles and median vote count for bundles, then use the remaining primary rules in order. Also rerun the primary ranking with vote thresholds of 500 and 5,000 and, separately, with a 6.5 rating anchor. Report who enters, leaves, or moves and whether the leading franchise changes. Recompute the group recommendation for each rating-anchor, vote-threshold, and installment-omission sensitivity. Use nominal per-film gross as a separate comparison with its known era and budget limitations.
- Produce one full candidate table per unit and window, including exclusions and reasons. Show raw and adjusted rating, votes, popularity, financial proxies, year and genre, observed package membership, rank, evidence for inclusion, and the main limitation. Compare shortlist membership and ranks by ID side by side. Label the recommendation and material signals **stable**, **changed**, or **insufficient evidence**.
- Implement the rule once and call it for both windows. Use vectorized checks and aggregations. Record the input and cache fingerprints, code revision, cache timestamp range, parameters, counts at each gate, and elapsed fetch and analysis times separately. Persist terminal fetch failures by ID and reason so Adjoa can distinguish a failed request from a missing TMDB title. Reuse the cached snapshot for an exact rerun. A fresh TMDB fetch can change the evidence. Check ranking ties, missing values, source-list changes, and top-to-bottom execution, then measure a cached run at approximately 500 titles.

### 6. Write the deal note and client memo

- Name the leading eligible lab bundle and its strongest supporting number with the window, denominator, and main limitation. Compare it with the next-best bundle and the strongest standalone alternative. Explain which bundle Kwame should negotiate for first and highlight any change in the leading franchise between windows. Use historical gross and production budget only to describe past theatrical performance. The lab supplies no licensing quotes, so Kwame will need prices to tell what fits within his $14M budget.
- Write one page in business language with the recommendation, both shortlists and their reasons, the two-window comparison, the deal note, and the cost of the remaining unknowns. Point Adjoa to the run record as the proposed improvement beyond the brief. It lets her reproduce a result and explain changes at the next review.

## Step 1 · Which films are in scope?

Read the current candidate CSV, the local cache, and the removal log using the paths in `config.toml`. Run `uv run --locked streamvault` from the project root if the candidate data need to be fetched. Running this notebook reads local files only.

Use the first occurrence of each numeric movie ID. For an uncached ID, the removal log can show a previous lookup miss for that ID even if its line has moved. An uncached ID with no logged lookup outcome stops the run after the audit table so Adjoa can investigate it.

Each check report shows what was tested and why it passed or failed. A failed check stops the run so an input problem cannot quietly change the recommendation.

In [1]:
import pandas as pd
from IPython.display import display

from streamvault.cache import iter_cached_records
from streamvault.config import CONFIG
from streamvault.diagnostics import raise_for_failed_checks, run_data_checks
from streamvault.load import DUPLICATE_ID_REASON, MovieId, iter_movie_id_rows
from streamvault.normalize import normalize_movie_records

source_path = CONFIG.source_file
# Keep a row for every entry in the CSV, including entries the parser skips.
source_audit = pd.DataFrame.from_records(
    (
        (row.source_line, row.value, None)
        if isinstance(row, MovieId)
        else (row.source_line, row.removed_value, row.reason)
        for row in iter_movie_id_rows(source_path)
    ),
    columns=["source_line", "source_value", "outcome"],
).astype({"source_value": "string", "outcome": "string"})

# Treat IDs like 00123 and 123 as the same film. Keep the first entry.
valid_id = source_audit["outcome"].isna()
source_audit["movie_key"] = source_audit["source_value"].where(valid_id).str.lstrip("0")
duplicate_id = valid_id & source_audit["movie_key"].duplicated()
source_audit.loc[duplicate_id, "outcome"] = DUPLICATE_ID_REASON
current_ids = set(source_audit.loc[source_audit["outcome"].isna(), "movie_key"])


# Read every cache entry so duplicate IDs are visible before we build the film table.
cache_entries = list(iter_cached_records(CONFIG.cache_file))
cached_keys = pd.Index([key.lstrip("0") for key, _ in cache_entries])
in_scope = cached_keys.isin(current_ids)
scoped_keys = cached_keys[in_scope]
scoped_records = [
    record for (_, record), belongs_to_list in zip(cache_entries, in_scope, strict=True)
    if belongs_to_list
]
duplicate_cache_keys = scoped_keys[scoped_keys.duplicated()].unique()
cache_timestamps = pd.to_datetime(
    pd.Series([record.get("cached_at_utc") for record in scoped_records], dtype="object"),
    format="ISO8601",
    errors="coerce",
    utc=True,
)
missing_cache_times = int(cache_timestamps.isna().sum())

load_stage = "After loading the current list and cache"
load_checks = run_data_checks(load_stage, {
    "Source line numbers are unique": (
        bool(source_audit["source_line"].is_unique),
        f"{len(source_audit)} source rows",
    ),
    "Cached movie IDs in scope are unique": (
        len(duplicate_cache_keys) == 0,
        f"{len(duplicate_cache_keys)} repeated IDs. Examples: {duplicate_cache_keys[:5].tolist()}",
    ),
    "Cached films in scope have readable timestamps": (
        missing_cache_times == 0,
        f"{missing_cache_times} missing or unreadable timestamps among {len(scoped_records)} records",
    ),
})
display(load_checks)
raise_for_failed_checks(load_stage, load_checks)

,check,status,details
0,Source line numbers are unique,PASS,127 source rows
1,Cached movie IDs in scope are unique,PASS,0 repeated IDs. Examples: []
2,Cached films in scope have readable timestamps,PASS,0 missing or unreadable timestamps among 112 r...


In [2]:
tables = normalize_movie_records(scoped_records)
movies = tables.movies
movie_ids = movies["movie_id"]
resolved_keys = set(movie_ids.astype("string"))
expected_keys = set(scoped_keys)

clean_stage = "After cleaning the film data"
checks = {
    "Each film has one unique movie ID": (
        bool(movie_ids.notna().all() and movie_ids.is_unique),
        f"{movie_ids.nunique()} distinct IDs across {len(movies)} film rows",
    ),
    "Cleaning keeps exactly the cached films in scope": (
        len(movies) == len(scoped_records) and resolved_keys == expected_keys,
        f"{len(movies)} film rows from {len(scoped_records)} cached entries",
    ),
}

# Missing amounts and zeros stay visible in the coverage report. Impossible amounts stop the run.
for field, label in {"budget": "Production budget", "revenue": "Box office gross"}.items():
    amounts = movies.reindex(columns=[field])[field]
    numeric_amounts = pd.to_numeric(amounts, errors="coerce")
    invalid_amount = amounts.notna() & ~numeric_amounts.between(0, float("inf"), inclusive="left").fillna(False)
    bad_ids = movie_ids.loc[invalid_amount]
    checks[f"{label} is finite and nonnegative when recorded"] = (
        bad_ids.empty,
        f"{len(bad_ids)} invalid values. Example movie IDs: {bad_ids.head(5).tolist()}",
    )

clean_checks = run_data_checks(clean_stage, checks)
display(clean_checks)
raise_for_failed_checks(clean_stage, clean_checks)

,check,status,details
0,Each film has one unique movie ID,PASS,112 distinct IDs across 112 film rows
1,Cleaning keeps exactly the cached films in scope,PASS,112 film rows from 112 cached entries
2,Production budget is finite and nonnegative wh...,PASS,0 invalid values. Example movie IDs: []
3,Box office gross is finite and nonnegative whe...,PASS,0 invalid values. Example movie IDs: []


In [3]:
# Match past lookup misses by ID so they still show up if a row moves.
lookup_reasons = pd.Series(dtype="string")
if CONFIG.removals_file.is_file() and CONFIG.removals_file.stat().st_size:
    removals = pd.read_csv(CONFIG.removals_file, dtype="string", keep_default_na=False)
    lookup_rows = removals.loc[
        removals["source_file"].eq(source_path.name)
        & removals["reason"].str.startswith("TMDB lookup: "),
        ["removed_value", "reason"],
    ].copy()
    lookup_rows["movie_key"] = lookup_rows["removed_value"].str.strip().str.lstrip("0")
    lookup_reasons = lookup_rows.groupby("movie_key")["reason"].agg(
        lambda reasons: " / ".join(sorted(set(reasons)))
    )

# A cached film takes priority over any earlier lookup miss.
pending = source_audit["outcome"].isna()
cached = pending & source_audit["movie_key"].isin(resolved_keys)
uncached = pending & ~cached
unexplained_reason = "No cached record or logged lookup outcome"
source_audit.loc[cached, "outcome"] = "Cached"
source_audit.loc[uncached, "outcome"] = (
    source_audit.loc[uncached, "movie_key"].map(lookup_reasons).fillna(unexplained_reason)
)

source_counts = source_audit["outcome"].value_counts().rename("source_rows")
unexplained_count = int(source_audit["outcome"].eq(unexplained_reason).sum())
source_stage = "After matching the source rows to the cache and removal log"
source_checks = run_data_checks(
    source_stage,
    {
        "Every source row has one outcome": (
            bool(source_audit["source_line"].is_unique and source_audit["outcome"].notna().all()),
            f"{len(source_audit)} source rows accounted for",
        ),
        "The film table preserves the resolved IDs": (
            len(movies) == int(cached.sum()) and resolved_keys == set(source_audit.loc[cached, "movie_key"]),
            f"{len(movies)} distinct films in scope",
        ),
        "Every uncached candidate has a logged outcome": (
            unexplained_count == 0,
            f"{unexplained_count} unexplained misses",
        ),
    },
)

# Show the audit before stopping on a failed check.
display(source_counts.to_frame())
display(source_audit.loc[~cached, ["source_line", "source_value", "outcome"]])
print(f"Cached IDs outside the current list: {cached_keys[~in_scope].nunique()}")
display(source_checks)
raise_for_failed_checks(source_stage, source_checks)

,source_rows
outcome,
Cached,112
Duplicate TMDB movie ID; skipped.,6
TMDB lookup: TMDB returned HTTP 404.,5
Not a positive numeric TMDB movie ID.,3
Blank row; no TMDB movie ID.,1


,source_line,source_value,outcome
16,18,8471293,TMDB lookup: TMDB returned HTTP 404.
28,30,-1,Not a positive numeric TMDB movie ID.
33,35,9412887,TMDB lookup: TMDB returned HTTP 404.
41,43,603,Duplicate TMDB movie ID; skipped.
58,60,278,Duplicate TMDB movie ID; skipped.
67,69,,Blank row; no TMDB movie ID.
71,73,8471294,TMDB lookup: TMDB returned HTTP 404.
88,90,1891,Duplicate TMDB movie ID; skipped.
89,91,155,Duplicate TMDB movie ID; skipped.
91,93,9412888,TMDB lookup: TMDB returned HTTP 404.


Cached IDs outside the current list: 0


,check,status,details
0,Every source row has one outcome,PASS,127 source rows accounted for
1,The film table preserves the resolved IDs,PASS,112 distinct films in scope
2,Every uncached candidate has a logged outcome,PASS,0 unexplained misses


### Release windows

Include a title in all years if it has a date and TMDB marks it as released. The 2000-onward window also requires a release date on or after 1 January 2000. A title without a collection ID is treated as a standalone. Collection counts cover the films observed in this list and show how membership changes at the cutoff.

In [4]:
cutoff = pd.Timestamp("2000-01-01")
candidates = movies.copy()
# Keep date and status columns even when no cached film supplies those fields.
candidates["release_date"] = movies.get(
    "release_date", pd.Series(pd.NaT, index=movies.index, dtype="datetime64[ns]")
)
candidates["status"] = movies.get(
    "status", pd.Series(pd.NA, index=movies.index, dtype="string")
)
candidates["title_type"] = candidates["collection_id"].notna().map(
    {True: "Collection-linked", False: "Standalone"}
)
candidates["release_year"] = candidates["release_date"].dt.year.astype("Int64")
candidates["decade"] = candidates["release_year"] // 10 * 10

dated = candidates["release_date"].notna()
released = candidates["status"].astype("string").eq("Released").fillna(False)
all_years = dated & released
# Use the same release rules in both windows and include 1 January 2000.
since_2000 = all_years & candidates["release_date"].ge(cutoff)

# Use a wide range. The newest cache timestamp keeps its upper end fixed for this snapshot.
first_plausible_year = 1880
snapshot_year = cache_timestamps.max().year
invalid_year = all_years & ~candidates["release_year"].between(
    first_plausible_year, snapshot_year
).fillna(False)
bad_year_ids = candidates.loc[invalid_year, "movie_id"]
year_detail = (
    f"Allowed years: {first_plausible_year} to {snapshot_year}. "
    f"{len(bad_year_ids)} out of range. Example movie IDs: {bad_year_ids.head(5).tolist()}"
    if len(cache_timestamps)
    else "No cached films in scope"
)
windows = {
    "All years": candidates.loc[all_years].copy(),
    "2000 onward": candidates.loc[since_2000].copy(),
}

status_counts = (
    candidates["status"].astype("string").fillna("Missing status")
    .value_counts().rename("films").rename_axis("release_status")
)
display(status_counts.to_frame())
print(f"Films missing a release date: {int((~dated).sum())}")

# Count films with no date first so each excluded film appears under one reason.
window_counts = {}
for window, films in windows.items():
    collection_sizes = films["collection_id"].value_counts()
    window_counts[window] = {
        "Resolved films": len(candidates),
        "Missing release date": int((~dated).sum()),
        "Dated and missing release status": int((dated & candidates["status"].isna()).sum()),
        "Dated and marked with another status": int((dated & candidates["status"].notna() & ~released).sum()),
        "Dated and released but before the cutoff": (
            int((all_years & ~since_2000).sum()) if window == "2000 onward" else 0
        ),
        "Films in window": len(films),
        "Collection-linked films": int(films["collection_id"].notna().sum()),
        "Standalone films": int(films["collection_id"].isna().sum()),
        "Observed collections": len(collection_sizes),
        "Collections with two or more observed films": int(collection_sizes.ge(2).sum()),
    }

display(pd.DataFrame(window_counts))

window_stage = "After applying the release windows"
window_checks = run_data_checks(window_stage, {
    "Dated releases have a plausible year": (bad_year_ids.empty, year_detail),
    "Each window accounts for all resolved films": (
        all(
            counts["Films in window"] + counts["Missing release date"]
            + counts["Dated and missing release status"]
            + counts["Dated and marked with another status"]
            + counts["Dated and released but before the cutoff"] == len(candidates)
            for counts in window_counts.values()
        ),
        f"{len(candidates)} films to account for in each window",
    ),
})
display(window_checks)
raise_for_failed_checks(window_stage, window_checks)

,films
release_status,
Released,112


Films missing a release date: 0


,All years,2000 onward
Resolved films,112,112
Missing release date,0,0
Dated and missing release status,0,0
Dated and marked with another status,0,0
Dated and released but before the cutoff,0,41
Films in window,112,71
Collection-linked films,96,63
Standalone films,16,8
Observed collections,32,24
Collections with two or more observed films,23,17


,check,status,details
0,Dated releases have a plausible year,PASS,Allowed years: 1880 to 2026. 0 out of range. E...
1,Each window accounts for all resolved films,PASS,112 films to account for in each window


In [5]:
decade_counts = pd.concat(
    {
        window: films.groupby(["decade", "title_type"]).size()
        for window, films in windows.items()
    },
    axis=1,
).fillna(0).astype(int).sort_index()
display(decade_counts)

collection_counts = pd.concat(
    {window: films["collection_id"].value_counts() for window, films in windows.items()},
    axis=1,
).fillna(0).astype(int).sort_index()
# Use the collection ID to attach its name. Names can change or be shared.
collection_ids_before = collection_counts.index
collection_counts = collection_counts.merge(
    tables.collections[["collection_id", "collection_name"]],
    left_index=True,
    right_on="collection_id",
    how="left",
    validate="one_to_one",
    indicator="collection_match",
).set_index("collection_id")

unmatched_collections = collection_counts.index[collection_counts["collection_match"].eq("left_only")]
join_stage = "After joining collection counts to collection names"
join_checks = run_data_checks(join_stage, {
    "The join keeps the same collection rows and IDs": (
        len(collection_counts) == len(collection_ids_before)
        and set(collection_counts.index) == set(collection_ids_before),
        f"{len(collection_ids_before)} rows before the join, {len(collection_counts)} after",
    ),
    "Every observed collection ID matches a reference row": (
        unmatched_collections.empty,
        f"{len(unmatched_collections)} unmatched IDs. Examples: {unmatched_collections[:5].tolist()}",
    ),
})
display(join_checks)
raise_for_failed_checks(join_stage, join_checks)
collection_counts = collection_counts.drop(columns="collection_match")
# The newer window is part of all years, so the difference gives the older count.
collection_counts["Before 2000"] = collection_counts["All years"] - collection_counts["2000 onward"]
cross_cutoff = collection_counts["Before 2000"].gt(0) & collection_counts["2000 onward"].gt(0)
print(f"Collections with observed films on both sides of the cutoff: {int(cross_cutoff.sum())}")
display(collection_counts[["collection_name", "All years", "2000 onward", "Before 2000"]])

All years  2000 onward
decade title_type                               
1960   Standalone                 1            0
1970   Collection-linked          5            0
1980   Collection-linked         12            0
       Standalone                 1            0
1990   Collection-linked         16            0
       Standalone                 6            0
2000   Collection-linked         42           42
       Standalone                 6            6
2010   Collection-linked         21           21
       Standalone                 2            2

,check,status,details
0,The join keeps the same collection rows and IDs,PASS,"32 rows before the join, 32 after"
1,Every observed collection ID matches a referen...,PASS,0 unmatched IDs. Examples: []


Collections with observed films on both sides of the cutoff: 8


,collection_name,All years,2000 onward,Before 2000
collection_id,,,,
10,Star Wars Collection,6,2,4
84,Indiana Jones Collection,4,1,3
119,The Lord of the Rings Collection,3,3,0
230,The Godfather Collection,3,0,3
263,The Dark Knight Collection,3,3,0
264,Back to the Future Collection,3,0,3
295,Pirates of the Caribbean Collection,5,5,0
304,Ocean's Collection,3,3,0
328,Jurassic Park Collection,3,1,2


### Outcome data coverage

Check which films have ratings, votes, popularity, budget, and gross in each window and title type. A reported zero is counted separately from a missing value. Step 2 will apply the scoring and financial validity rules. Missing ratings or votes are listed here so Step 2 can show why each film was excluded. Step 2 must check that every film it actually scores has both inputs.

In [6]:
outcome_fields = ["vote_average", "vote_count", "popularity", "budget", "revenue"]
title_types = ["Collection-linked", "Standalone"]
coverage_tables = {}

for window, films in windows.items():
    # Keep expected fields in the report even when every value is missing.
    outcomes = films.reindex(columns=outcome_fields)
    groups = films["title_type"]
    totals = groups.value_counts().reindex(title_types, fill_value=0)
    missing = outcomes.isna().groupby(groups).sum().reindex(title_types, fill_value=0)
    zeros = outcomes.eq(0).groupby(groups).sum().reindex(title_types, fill_value=0)
    # Show one row per field and title type.
    coverage_tables[window] = pd.concat(
        {
            "Films": pd.DataFrame({field: totals for field in outcome_fields}),
            "Recorded": missing.rsub(totals, axis=0),
            "Missing": missing,
            "Reported zero": zeros,
        },
        axis=1,
    ).stack(level=1)

outcome_coverage = pd.concat(coverage_tables, names=["window", "title_type", "field"])
display(outcome_coverage)

Films  ...  Reported zero
window      title_type        field                ...               
All years   Collection-linked vote_average     96  ...              0
                              vote_count       96  ...              0
                              popularity       96  ...              0
                              budget           96  ...              0
                              revenue          96  ...              0
            Standalone        vote_average     16  ...              0
                              vote_count       16  ...              0
                              popularity       16  ...              0
                              budget           16  ...              0
                              revenue          16  ...              0
2000 onward Collection-linked vote_average     63  ...              0
                              vote_count       63  ...              0
                              popularity       63  ...              0
                              budget           63  ...              0
                              revenue          63  ...              0
            Standalone        vote_average      8  ...              0
                              vote_count        8  ...              0
                              popularity        8  ...              0
                              budget            8  ...              0
                              revenue           8  ...              0

[20 rows x 4 columns]

### Step 1 conclusion

**Ready for Step 2.** The checks passed at loading, cleaning, source matching, window selection, and the collection join. Every film in both windows has a recorded rating and vote count.

- **The candidate list is accounted for.** Of 127 rows, 112 gave us distinct films, 6 were repeats, 5 returned a “not found” response from TMDB, 3 had invalid IDs, and 1 was blank. All 112 films have release dates and are marked as released. We have no film evidence for the nine rows lost to lookup or input problems, so the shortlist can only reflect the films we resolved.
- **The 2000 cutoff changes the choices.** It removes 41 films and halves the standalone pool from 16 to 8. Filling five standalone spots would use five of those eight before any scoring exclusions. Collections with at least two films in the list fall from 23 to 17. Eight collections have films on both sides of the cutoff. For example, Indiana Jones goes from four observed films to one, so it falls below the two-film minimum in the newer window.
- **Step 2 will check who qualifies.** Ratings, vote counts, popularity, production budget, and reported box office gross are present and nonzero for every film in both windows. We can now check the values, apply the 1,000-vote minimum, and rank the eligible titles and collections in each window. The franchise-versus-standalone comparison follows in Step 3.